[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bigdata-com/bigdata-getting-started/blob/main/notebooks/03_search.ipynb)

# 03 · Search — find the most relevant content

The **[Bigdata.com](https://bigdata.com)** Search Service (`POST /v1/search`)
retrieves the most relevant passages ("chunks") from a premium corpus of news,
earnings-call transcripts, filings, and investment research — plus your own
uploaded content — with optional live web results.

**What it's good for**
- Powering RAG pipelines and agents with accurate, sourced, real-time context
- Monitoring companies/themes across thousands of trusted sources
- Due diligence and event research with precise entity, topic and date filters

**Two modes**
- **`fast`** — runs a single query with exactly the filters you specify. Best
  when you control the filters (programmatic pipelines).
- **`smart`** — analyses the query text, auto-derives filters, and runs several
  sub-queries for better coverage. Best for raw user questions. (In smart mode
  only `timestamp` and `source` filters are allowed.)

In [ ]:
!pip install -q requests env-colab-pass

In [ ]:
import requests
from env_colab_pass import passutil

# Your API key is read from the BIGDATA_API_KEY environment variable.
# Never hard-code keys in a notebook you plan to share or commit.
API_KEY = passutil.get_secret_value("BIGDATA_API_KEY")

# Bigdata exposes two hosts:
#   - api.bigdata.com    -> Knowledge Graph & Search Service (deterministic APIs)
#   - agents.bigdata.com -> Research Agent & Workflows (AI agents, streamed)
API_BASE = "https://api.bigdata.com"
AGENTS_BASE = "https://agents.bigdata.com"

# The key must be sent in the X-API-KEY header on every request.
HEADERS = {"X-API-KEY": API_KEY, "Content-Type": "application/json"}

A small helper so the examples stay readable. It prints the top chunks and the units consumed.

In [ ]:
def search(query_text=None, mode="fast", max_chunks=5, show=3, **filters):
    """Call POST /v1/search. Pass filters=<dict> via the `filters` kwarg."""
    query = {"max_chunks": max_chunks}
    if query_text is not None:
        query["text"] = query_text
    if filters.get("filters"):
        query["filters"] = filters["filters"]
    payload = {"search_mode": mode, "query": query}
    r = requests.post(f"{API_BASE}/v1/search", headers=HEADERS, json=payload, timeout=90)
    r.raise_for_status()
    data = r.json()
    print(f"{len(data['results'])} documents · {data['usage']['api_query_units']} query units")
    for doc in data["results"][:show]:
        ch = doc["chunks"][0]
        print(f"\n\u2022 {doc['headline']}  [{doc['source']['name']}, {doc['timestamp'][:10]}]")
        print(f"  sentiment={ch['sentiment']:+.2f}  {ch['text'][:200]}...")
    return data

### Example 1 — Semantic search (fast)
Just describe what you want in natural language.

In [ ]:
_ = search("government debt to GDP ratio outlook", max_chunks=5)

### Example 2 — Filter by entity
Target a specific company with its Knowledge Graph entity ID (see notebook 02).
`any_of` matches chunks mentioning any of the listed entities.

In [ ]:
# Resolve Apple's entity ID on the fly.
# Look up Apple Inc. by ISIN 
response = requests.post(
    f"{API_BASE}/v1/knowledge-graph/companies/isin",
    headers=HEADERS,
    json={"values": ["US0378331005"]},
    timeout=30
)
apple = response.json()["results"]["US0378331005"]["id"]

                 
print(apple_id)

_ = search("supply chain and manufacturing", max_chunks=5,
           filters={"entity": {"any_of": [apple]}})

### Example 3 — Filter by keyword
Require specific terms to appear. `all_of` / `any_of` / `none_of` are supported,
and you can restrict to `HEADLINE`, `BODY`, or `ALL`.

In [ ]:
_ = search("financial institutions sanctions compliance obligations", max_chunks=5,
           filters={"keyword": {"any_of": ["Sanctions", "Compliance"]}})

### Example 4 — Filter by sentiment
Every chunk has a market-impact sentiment score from -1.0 to 1.0. Filter by
explicit ranges (here: clearly negative).

In [ ]:
_ = search("climate transition risk for banks", max_chunks=5,
           filters={"sentiment": {"ranges": [{"min": -1.0, "max": -0.1}]}})

### Example 5 — Filter by date range
Restrict to a publication window (ISO 8601, UTC).

In [ ]:
_ = search("interest rate policy outlook", max_chunks=5,
           filters={"timestamp": {"start": "2025-01-01T00:00:00Z", "end": "2025-12-31T23:59:59Z"}})

### Example 6 — Filter by document type
Narrow to a content type, e.g. only earnings-call transcripts.

In [ ]:
_ = search("Coreweave management guidance for next quarter", max_chunks=5,
           filters={"document_type": {"mode": "INCLUDE", "values": [{"type": "TRANSCRIPT"}]}})

### Example 7 — Smart search
Send a raw question and let Bigdata.com derive the filters and run sub-queries.

In [ ]:
_ = search("How is Nvidia exposed to US-China export restrictions?", mode="smart", max_chunks=8)

## Fast vs. Smart — quick guide
| | fast | smart |
|---|---|---|
| Filters | all filters, exactly as given | only `timestamp` & `source` (others auto-derived) |
| Sub-queries | one | several, for coverage |
| Best for | controlled pipelines | Invoked by Agent and raw user questions |

## Batch Search
Need to run **many** queries efficiently (e.g. a whole watchlist or a screen)?
Use the **Batch Search API** (`/v1/search/batches`). See the cookbook:
👉 https://github.com/Bigdata-com/bigdata-cookbook/tree/main/Batch_Search_API

**Next:** [`04_research.ipynb`](04_research.ipynb)

_Source: [Bigdata.com](https://bigdata.com) · Search docs: https://docs.bigdata.com/api-reference/search/search-documents_